# S14 - Híbridos y guardarraíles (Pagarium)

**Objetivo (RA5-b/d/e):** extraer reglas de los datos (**FIGS**) y proteger un agente LLM con **guardarraíles** de reglas. Cierre: neuro-simbólico, AI Act y proyecto Pagarium.

### Python en un vistazo (lo justo para esta sesión)

| Lo que verás | Qué significa |
|---|---|
| `# texto` | **Comentario**: Python lo ignora; es una nota para ti |
| `nombre = valor` | **Guarda** un valor con un nombre |
| `{ "clave": "valor" }` | Un **diccionario**: parejas nombre → valor |
| `[a, b, c]` | Una **lista**: valores ordenados |
| `def nombre(...):` | Crea una **función** |
| `"texto"` / `'texto'` | Un **texto** (string), entre comillas |
| `==` / `<` / `>=` | Comparaciones: igual / menor / mayor o igual |
| `and` / `or` | Y lógico / O lógico |
| `if ...:` / `for ... in ...:` | Condición / repetición |
| `print(...)` | Escribe en pantalla |

Ejecuta las celdas de arriba a abajo; los comentarios `#` explican cada línea.

## 1. FIGS redescubre la política latente de Pagarium

Generamos 4.000 pagos con una regla oculta y un 3 % de ruido; **FIGS** la recupera de forma legible (reglas, no una caja negra).

In [ ]:
%pip install imodels scikit-learn
import numpy as np                        # numpy = números y matrices; np.random genera aleatorios
from imodels import FIGSClassifier        # FIGS: genera reglas interpretables
from sklearn.model_selection import train_test_split   # separa datos en entrenar/probar

rng = np.random.default_rng(0)            # semilla fija: siempre los mismos datos
n = 4000                                  # número de pagos simulados
importe = rng.uniform(0, 2000, n)         # importes entre 0 y 2000
antiguedad = rng.uniform(0, 24, n)        # meses de antigüedad
n_intentos = rng.integers(0, 6, n)        # intentos: 0..5

# La política LATENTE (oculta): rechazar si (importe>1000 y antigüedad<6) o n_intentos>=4
y = ((importe > 1000) & (antiguedad < 6)) | (n_intentos >= 4)
flip = rng.random(n) < 0.03               # 3 % de ruido: volcamos algunas etiquetas
y = np.where(flip, ~y, y).astype(int)     # 0/1 en vez de True/False

X = np.column_stack([importe, antiguedad, n_intentos])   # junta las 3 columnas
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)  # 70% entrenar / 30% probar

clf = FIGSClassifier(max_rules=6)         # como mucho 6 reglas
clf.fit(X_tr, y_tr)                       # APRENDE las reglas a partir de los datos
print("Accuracy:", round(clf.score(X_te, y_te), 3))   # acierto sobre los datos de prueba
print(clf)                                # imprime el árbol de reglas (legible)

## 2. Guardarraíl de un agente LLM

Las reglas **validan y acotan** lo que propone un agente antes de ejecutarlo. Aquí simulamos la salida de un LLM con un texto.

In [ ]:
import re   # re = expresiones regulares (buscar patrones en texto)

def validar_accion_llm(texto):            # recibe la propuesta del agente y devuelve (ok, motivo)
    # findall(r"(\d+(?:\.\d+)?)\s*EUR", ...)  busca números seguidos de EUR
    importes = [float(x) for x in re.findall(r"(\d+(?:\.\d+)?)\s*EUR", texto)]
    if any(i > 1000 for i in importes):   # any(...) = "¿alguno cumple?"
        return False, "importe fuera de rango"
    if "cuenta no verificada" in texto.lower():   # .lower() pasa a minúsculas
        return False, "destino no permitido"
    if re.search(r"transferir.*sin confirmar", texto, re.IGNORECASE):  # patrón de texto
        return False, "requiere confirmacion humana"
    return True, "ok"

for t in ["Transferir 500 EUR a la cuenta A",   # ok
          "Transferir 5000 EUR",                 # fuera de rango
          "Pagar a cuenta no verificada"]:       # destino no permitido
    print(t, "->", validar_accion_llm(t))

### Un guardarraíl con más reglas

Añadimos importe mínimo y bloqueo de divisas.

In [ ]:
def validar_accion_llm_extendido(texto):
    importes = [float(x) for x in re.findall(r"(\d+(?:\.\d+)?)\s*EUR", texto)]
    if any(i > 1000 for i in importes):
        return False, "importe fuera de rango"
    if any(i <= 0 for i in importes):          # importe no positivo
        return False, "importe no valido"
    if "cuenta no verificada" in texto.lower():
        return False, "destino no permitido"
    if re.search(r"transferir.*sin confirmar", texto, re.IGNORECASE):
        return False, "requiere confirmacion humana"
    divisas = re.findall(r"\b(USD|GBP|JPY|CHF)\b", texto.upper())   # busca divisas no EUR
    if divisas:
        return False, f"divisa no permitida: {divisas[0]}"
    return True, "ok"

for t in ["Transferir 500 EUR a la cuenta A",
          "Transferir 5000 EUR",
          "Transferir 0 EUR",
          "Pagar a cuenta no verificada",
          "Transferir 300 USD a la cuenta B"]:
    print(t, "->", validar_accion_llm_extendido(t))

### FIGS con más reglas permitidas

`max_rules` más alto deja que FIGS recupere más reglas.

In [ ]:
clf2 = FIGSClassifier(max_rules=10)   # hasta 10 reglas
clf2.fit(X_tr, y_tr)
print("Accuracy (max_rules=10):", round(clf2.score(X_te, y_te), 3))
print(clf2)

## 3. Actividad — las dos capas del proyecto Pagarium

1. **Capa de negocio:** implementa la tabla DMN (S11) con `rule-engine` o `MicroMotor2`.
2. **Capa guardarraíl:** añade una regla nueva a `validar_accion_llm` (p. ej. bloquear divisas no permitidas).
3. Escribe un informe de 3 líneas: ¿usarías un motor de reglas aquí? Justifícalo con el benchmark.

In [ ]:
# TODO: capa de negocio + guardarrail + conclusion
...

## 4. Cierre — neuro-simbólico y AI Act

- **Neuro-simbólico:** red (aprende) + reglas (acotan y explican) → menos alucinaciones y más trazabilidad.
- **AI Act:** alto riesgo del Anexo III (scoring) aplazado a dic-2027 por el Ómnibus Digital; transparencia y alfabetización vigentes desde ago-2026. **Verifica el calendario antes de evaluar.**

**Para casa:** ¿por qué un guardarraíl de reglas mejora la confianza en un agente LLM?